# Notebook 2 - Training a Truth Probe on One Dataset

## Goal

This notebook runs the first complete experiment. We train one linear probe on the `repeng_truthful` dataset and evaluate it on train, validation, and test splits.

The purpose is not yet to compare many LLMs. The purpose is to verify that the white-box probing pipeline works end to end on the current default model.


## Step 1 - Import the project package

The setup cell finds the repository root and makes `src/lie_detector_llm` importable. This makes the notebook work whether it is launched from the repository root or from the `notebooks` directory.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "lie_detector_llm").exists():
            return candidate
    raise RuntimeError("Could not find the project root. Run this notebook from the repository.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")


## Step 2 - Build the dataset collection

We use the lightweight local datasets in this notebook. The key dataset is `repeng_truthful`, which comes from the RepEng-style truthfulness JSONL file stored under `data/raw/repeng/truthful.jsonl`.

Each `repeng_truthful` group contains one honest self-report statement and one dishonest self-report statement. The probe learns to rank the honest statement above the dishonest one.


In [ ]:
from lie_detector_llm.datasets import build_dataset_collection

INCLUDE_HF_DATASETS = False

collection = build_dataset_collection(include_hf_datasets=INCLUDE_HF_DATASETS)
frame = collection.subset("repeng_truthful")

print(f"Available datasets: {collection.dataset_names()}")
print(f"Selected dataset : repeng_truthful")
print(f"Prompt rows      : {len(frame)}")
print(f"Question groups  : {frame['group_id'].nunique()}")
display(frame.head())


## Step 3 - Choose the model, layer, and probe

The current project baseline is `microsoft/phi-2`.

Phi-2 is a 2.7B parameter model with 32 transformer layers. It is the current local baseline, but still much smaller than the Llama 70B models requested for the final comparison.

We start with:

- model: `microsoft/phi-2`,
- probe: logistic regression (`lr`),
- layer: `-1`, meaning the final transformer layer.

The final layer is a simple first baseline. Notebook 4 will test every layer.


In [ ]:
MODEL_NAME = "microsoft/phi-2"
PROBE_METHOD = "lr"
LAYER_INDEX = -1
LOAD_IN_4BIT = False
ACTIVATION_BATCH_SIZE = 2

print("Experiment configuration")
print(f"Model          : {MODEL_NAME}")
print(f"Probe          : {PROBE_METHOD}")
print(f"Layer index    : {LAYER_INDEX}")
print(f"4-bit loading  : {LOAD_IN_4BIT}")


## Step 4 - Run the probe experiment

The function below performs the full pipeline:

1. split groups into train, validation, and test,
2. extract hidden activations from the selected model,
3. select the requested layer,
4. train the probe on the train split,
5. evaluate grouped accuracy on all splits.

No text is generated by the model. We only run forward passes and read hidden states.


In [ ]:
from lie_detector_llm.experiment import run_probe_experiment

results = run_probe_experiment(
    frame=frame,
    model_name=MODEL_NAME,
    probe_method=PROBE_METHOD,
    layer_index=LAYER_INDEX,
    activation_batch_size=ACTIVATION_BATCH_SIZE,
    load_in_4bit=LOAD_IN_4BIT,
    show_progress=True,
)

result_table = results.summary_table()
display(result_table)


## Step 5 - Interpret grouped accuracy

A grouped accuracy of `1.00` means the probe selected the true candidate in every group. A grouped accuracy of `0.50` on a two-candidate dataset is roughly chance level.

Important interpretation:

- high train accuracy means the probe can fit the training examples,
- high validation/test accuracy means the signal generalizes to unseen groups from the same dataset,
- a large train-test gap suggests overfitting.


## Step 6 - Compare all probe methods on the same dataset

Logistic regression is only one probe. The RepEng style of analysis compares several linear methods because different probes can generalize differently.

The next cell trains `dim`, `lat`, `lr`, and `pca-g` on the same dataset and reports the test split accuracy.


In [ ]:
import pandas as pd

rows = []
for method in ["dim", "lat", "lr", "pca-g"]:
    out = run_probe_experiment(
        frame=frame,
        model_name=MODEL_NAME,
        probe_method=method,
        layer_index=LAYER_INDEX,
        activation_batch_size=ACTIVATION_BATCH_SIZE,
        load_in_4bit=LOAD_IN_4BIT,
        show_progress=True,
    )
    method_rows = out.summary_table()
    method_rows["probe_method"] = method
    rows.append(method_rows)

comparison = pd.concat(rows, ignore_index=True)
test_comparison = (
    comparison[comparison["split"] == "test"]
    .sort_values("grouped_accuracy", ascending=False)
    .reset_index(drop=True)
)

display(test_comparison[["probe_method", "model_name", "layer_index", "grouped_accuracy"]])


## Conclusion

This notebook answers the first technical question: can the repository train and evaluate a truth probe on one dataset?

The next notebook asks the more important scientific question: does a probe trained on one dataset transfer to different datasets, or does it only memorize dataset-specific patterns?
